# sequence_risk_reward_feature_screen_v1

Purpose: turn the repeated `wf_2022` sequence-risk drawdown miss into a reusable reward-feature recipe before any longer PPO run.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1` runtime composer.
- Direct comparison baselines: `cap10_sequence_risk_checkpoint_penalty_screen_v1`, `cap10_sequence_risk_target_cap_bc_screen_v1`, and `cap10_bc_sequence_risk_state_diagnostic_v1`.
- Role: `diagnose` / reward recipe screen; no actor is promoted or retained by this notebook.
- Changed layer: `reward` and reusable `src` capability only. This notebook does not train a new actor.
- Target: excess return `>= +30pp` and MaxDD `<= 15%` on replay folds.

Historical reflection:
- Closest success: no-sequence-cap BC remains the safest two-fold reference but misses the MaxDD gate by about `3.40pp`.
- Closest failure: direct target caps (`trend_break_any_cap10`, `slope_low_cap15`) improved return or one fold while failing robust drawdown; checkpoint penalty still found no full-gate candidate.
- Non-duplicate: this run does not cap target labels again and does not rerun PPO. It converts the mechanism into a default-off reward feature plus fold-local screen so training can learn risk before position changes rather than being cut by a post-hoc cap.

External basis:
- Qlib encourages reproducible research workflows and artifact-first iteration: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- PPO can drift under policy updates, so reward/selection changes should be screened before longer runs: https://arxiv.org/abs/1707.06347
- Portfolio RL needs walk-forward / out-of-sample validation and risk-aware objectives: https://arxiv.org/abs/2006.04779 and https://arxiv.org/abs/1712.06924
- Multiple-test / selection risk remains a concern when adding reward knobs: https://doi.org/10.21314/JCF.2016.322


In [1]:
from __future__ import annotations

import itertools
import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.sequence_risk_objective import (
    SequenceRiskPathConfig,
    apply_sequence_risk_mask,
    score_sequence_risk_path,
)

ROOT = Path.cwd()
if not (ROOT / 'runtime').exists():
    raise RuntimeError(f'Notebook must run from repo root, got {ROOT}')

EXPERIMENT = 'sequence_risk_reward_feature_screen_v1'
CARD = ROOT / 'runtime' / 'reward' / 'card' / EXPERIMENT
SOURCE_SELECTOR_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_sequence_risk_checkpoint_penalty_screen_v1'
TARGET_CAP_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_sequence_risk_target_cap_bc_screen_v1'
SEQUENCE_DIAGNOSTIC_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_bc_sequence_risk_state_diagnostic_v1'

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 15.0
POSITION_REFERENCE = 0.10

CARD.mkdir(parents=True, exist_ok=True)
print('card', CARD)


card /Users/rziiiii/.openclaw/workspace/runtime/reward/card/sequence_risk_reward_feature_screen_v1


In [2]:
selector_selection = json.loads((SOURCE_SELECTOR_CARD / 'selection.json').read_text())
target_cap_selection = json.loads((TARGET_CAP_CARD / 'selection.json').read_text())
sequence_selection = json.loads((SEQUENCE_DIAGNOSTIC_CARD / 'selection.json').read_text())
selector_scores = pd.read_csv(SOURCE_SELECTOR_CARD / 'candidate_penalty_scores.csv')
sequence_cap_audit = pd.read_csv(TARGET_CAP_CARD / 'sequence_risk_cap_audit.csv')

source_context = {
    'selector_decision': selector_selection.get('decision'),
    'selector_selected_candidate': selector_selection.get('selected_candidate_id'),
    'selector_selected_max_dd': selector_selection.get('selected_max_max_dd_pct'),
    'selector_full_gate_candidate_count': selector_selection.get('full_gate_candidate_count'),
    'target_cap_decision': target_cap_selection.get('decision'),
    'sequence_diagnostic_decision': sequence_selection.get('decision'),
}
source_context


{'selector_decision': 'sequence_risk_checkpoint_penalty_confirms_no_safe_policy_improvement_no_promotion',
 'selector_selected_candidate': 'sequence_target_cap_bc::baseline_no_sequence_cap',
 'selector_selected_max_dd': 18.39642924161966,
 'selector_full_gate_candidate_count': 0,
 'target_cap_decision': 'sequence_risk_target_cap_bc_no_robust_improvement_no_promotion',
 'sequence_diagnostic_decision': 'sequence_risk_state_candidate_ready_for_target_screen_no_promotion'}

In [3]:
def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [json_safe(v) for v in value]
    if hasattr(value, 'item'):
        try:
            return json_safe(value.item())
        except (TypeError, ValueError):
            pass
    try:
        if value is None or pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def thresholds_for_fold(fold: str, *, candidate: str = 'trend_break_any_cap10') -> dict[str, dict[str, Any]]:
    row = sequence_cap_audit[
        sequence_cap_audit['candidate'].eq(candidate)
        & sequence_cap_audit['fold'].eq(f'{fold}_train')
    ]
    if row.empty:
        return {}
    return json.loads(str(row.iloc[0]['thresholds_json'] or '{}'))


def daily_path(candidate: str, fold: str) -> Path:
    return TARGET_CAP_CARD / f'{candidate}_{fold}_daily_path.csv'


def normalise_drawdown_fraction(daily: pd.DataFrame) -> pd.Series:
    if 'drawdown' in daily.columns:
        dd = pd.to_numeric(daily['drawdown'], errors='coerce').fillna(0.0).abs()
        return dd / 100.0 if float(dd.max()) > 1.5 else dd
    nav = pd.to_numeric(daily['strategy_nav'], errors='coerce')
    return (1.0 - nav / nav.cummax().replace(0.0, np.nan)).fillna(0.0).abs()


def reward_penalty_burden(
    daily: pd.DataFrame,
    thresholds: dict[str, dict[str, Any]],
    *,
    mode: str,
    position_threshold: float,
    position_penalty: float,
    drawdown_threshold: float,
    drawdown_penalty: float,
) -> dict[str, float]:
    frame = daily.copy()
    frame['datetime'] = pd.to_datetime(frame['datetime'])
    frame = frame.dropna(subset=['datetime']).sort_values('datetime')
    frame.index = pd.DatetimeIndex(frame['datetime'])
    frame.index.name = None
    mask, _ = apply_sequence_risk_mask(frame, thresholds, mode=mode)
    position = pd.to_numeric(frame['position'], errors='coerce').fillna(0.0)
    drawdown = normalise_drawdown_fraction(frame)
    penalty = mask.astype(float) * (
        np.maximum(position - position_threshold, 0.0) * position_penalty
        + np.maximum(drawdown - drawdown_threshold, 0.0) * drawdown_penalty
    )
    breach = drawdown > drawdown_threshold
    return {
        'mean_penalty': float(penalty.mean()),
        'mean_penalty_on_sequence': float(penalty[mask].mean()) if bool(mask.any()) else 0.0,
        'mean_penalty_on_breach': float(penalty[breach].mean()) if bool(breach.any()) else 0.0,
        'max_penalty': float(penalty.max()) if len(penalty) else 0.0,
        'active_penalty_rows': int((penalty > 0.0).sum()),
    }


In [4]:
modes = ['any', 'two_of_three', 'all']
path_config = SequenceRiskPathConfig(target_max_dd_pct=TARGET_MAX_DD_PCT, sequence_position_reference=POSITION_REFERENCE)
mode_rows = []
for candidate in ['baseline_no_sequence_cap', 'trend_break_any_cap10']:
    for fold in ['wf_2022', 'wf_2023_2024']:
        path = daily_path(candidate, fold)
        if not path.exists():
            continue
        daily = pd.read_csv(path)
        thresholds = thresholds_for_fold(fold)
        for mode in modes:
            metrics = score_sequence_risk_path(daily, thresholds=thresholds, mode=mode, config=path_config)
            mode_rows.append({
                'candidate': candidate,
                'fold': fold,
                'mode': mode,
                'min_feature_hits': {'any': 1, 'two_of_three': 2, 'all': 3}[mode],
                'daily_path': str(path.relative_to(ROOT)),
                **metrics,
            })

sequence_risk_mode_scores = pd.DataFrame(mode_rows)
sequence_risk_mode_scores.to_csv(CARD / 'sequence_risk_mode_scores.csv', index=False)
sequence_risk_mode_scores.sort_values(['candidate', 'fold', 'min_feature_hits'])


,candidate,fold,mode,min_feature_hits,daily_path,rows,sequence_rows,sequence_share,sequence_avg_pos,non_sequence_avg_pos,sequence_position_lift,sequence_max_dd_pct,drawdown_breach_rows,sequence_breach_rows,sequence_breach_capture,sequence_hit_count_mean,sequence_exposure_loss,sequence_drawdown_loss,sequence_breach_capture_loss,sequence_path_penalty
0,baseline_no_sequence_cap,wf_2022,any,1,runtime/model_selection/card/cap10_sequence_ri...,250,225,0.900,0.160863,0.042335,0.118528,18.396429,30,30,1.000000,1.676,0.608626,0.226429,0.608626,0.455747
1,baseline_no_sequence_cap,wf_2022,two_of_three,2,runtime/model_selection/card/cap10_sequence_ri...,250,188,0.752,0.177759,0.061836,0.115923,18.396429,30,30,1.000000,1.676,0.777586,0.226429,0.777586,0.557123
2,baseline_no_sequence_cap,wf_2022,all,3,runtime/model_selection/card/cap10_sequence_ri...,250,6,0.024,0.165549,0.148603,0.016946,14.809509,30,0,0.000000,1.676,0.655487,0.000000,0.000000,0.229420
3,baseline_no_sequence_cap,wf_2023_2024,any,1,runtime/model_selection/card/cap10_sequence_ri...,251,0,0.000,0.000000,0.145913,-0.145913,0.000000,0,0,0.000000,0.000,0.000000,0.000000,0.000000,0.000000
4,baseline_no_sequence_cap,wf_2023_2024,two_of_three,2,runtime/model_selection/card/cap10_sequence_ri...,251,0,0.000,0.000000,0.145913,-0.145913,0.000000,0,0,0.000000,0.000,0.000000,0.000000,0.000000,0.000000
5,baseline_no_sequence_cap,wf_2023_2024,all,3,runtime/model_selection/card/cap10_sequence_ri...,251,0,0.000,0.000000,0.145913,-0.145913,0.000000,0,0,0.000000,0.000,0.000000,0.000000,0.000000,0.000000
6,trend_break_any_cap10,wf_2022,any,1,runtime/model_selection/card/cap10_sequence_ri...,250,225,0.900,0.135131,0.042335,0.092796,19.550984,46,46,1.000000,1.676,0.351310,0.303399,0.351310,0.332146
7,trend_break_any_cap10,wf_2022,two_of_three,2,runtime/model_selection/card/cap10_sequence_ri...,250,188,0.752,0.147988,0.058728,0.089260,19.550984,46,46,1.000000,1.676,0.479878,0.303399,0.479878,0.409286
8,trend_break_any_cap10,wf_2022,all,3,runtime/model_selection/card/cap10_sequence_ri...,250,6,0.024,0.166364,0.124855,0.041509,16.367717,46,2,0.043478,1.676,0.663637,0.091181,0.028854,0.275959
9,trend_break_any_cap10,wf_2023_2024,any,1,runtime/model_selection/card/cap10_sequence_ri...,251,0,0.000,0.000000,0.149038,-0.149038,0.000000,0,0,0.000000,0.000,0.000000,0.000000,0.000000,0.000000


In [5]:
baseline_mode_scores = sequence_risk_mode_scores[
    sequence_risk_mode_scores['candidate'].eq('baseline_no_sequence_cap')
].copy()
mode_summary = []
for mode, group in baseline_mode_scores.groupby('mode'):
    wf2022 = group[group['fold'].eq('wf_2022')].iloc[0].to_dict() if not group[group['fold'].eq('wf_2022')].empty else {}
    wf2024 = group[group['fold'].eq('wf_2023_2024')].iloc[0].to_dict() if not group[group['fold'].eq('wf_2023_2024')].empty else {}
    capture_loss = max(0.0, 1.0 - float(wf2022.get('sequence_breach_capture', 0.0))) * 5.0
    breadth_loss = float(wf2022.get('sequence_share', 0.0)) * 0.30
    over_breadth_loss = max(0.0, float(wf2022.get('sequence_share', 0.0)) - 0.80) * 1.00
    oos_overreach_loss = float(wf2024.get('sequence_share', 0.0)) * 2.0
    loss = capture_loss + breadth_loss + over_breadth_loss + oos_overreach_loss
    mode_summary.append({
        'mode': mode,
        'min_feature_hits': {'any': 1, 'two_of_three': 2, 'all': 3}[mode],
        'wf_2022_sequence_share': float(wf2022.get('sequence_share', 0.0)),
        'wf_2022_breach_capture': float(wf2022.get('sequence_breach_capture', 0.0)),
        'wf_2022_sequence_avg_pos': float(wf2022.get('sequence_avg_pos', 0.0)),
        'wf_2023_2024_sequence_share': float(wf2024.get('sequence_share', 0.0)),
        'capture_loss': capture_loss,
        'breadth_loss': breadth_loss,
        'over_breadth_loss': over_breadth_loss,
        'oos_overreach_loss': oos_overreach_loss,
        'mode_loss': loss,
    })
mode_summary = pd.DataFrame(mode_summary).sort_values(['mode_loss', 'min_feature_hits']).reset_index(drop=True)
selected_mode_row = mode_summary.iloc[0].to_dict()
selected_mode = str(selected_mode_row['mode'])
selected_min_hits = int(selected_mode_row['min_feature_hits'])
mode_summary.to_csv(CARD / 'sequence_risk_mode_summary.csv', index=False)
mode_summary


,mode,min_feature_hits,wf_2022_sequence_share,wf_2022_breach_capture,wf_2022_sequence_avg_pos,wf_2023_2024_sequence_share,capture_loss,breadth_loss,over_breadth_loss,oos_overreach_loss,mode_loss
0,two_of_three,2,0.752,1.0,0.177759,0.0,0.0,0.2256,0.0,0.0,0.2256
1,any,1,0.900,1.0,0.160863,0.0,0.0,0.2700,0.1,0.0,0.3700
2,all,3,0.024,0.0,0.165549,0.0,5.0,0.0072,0.0,0.0,5.0072


In [6]:
grid_rows = []
position_thresholds = [0.10, 0.12, 0.15]
position_penalties = [1.5, 2.0, 3.0, 4.0]
drawdown_penalties = [1.0, 2.0, 3.0]
drawdown_threshold = TARGET_MAX_DD_PCT / 100.0
for position_threshold, position_penalty, drawdown_penalty in itertools.product(
    position_thresholds,
    position_penalties,
    drawdown_penalties,
):
    fold_burdens = []
    for fold in ['wf_2022', 'wf_2023_2024']:
        path = daily_path('baseline_no_sequence_cap', fold)
        daily = pd.read_csv(path)
        burden = reward_penalty_burden(
            daily,
            thresholds_for_fold(fold),
            mode=selected_mode,
            position_threshold=position_threshold,
            position_penalty=position_penalty,
            drawdown_threshold=drawdown_threshold,
            drawdown_penalty=drawdown_penalty,
        )
        fold_burdens.append((fold, burden))
    wf2022 = dict(fold_burdens[0][1])
    wf2024 = dict(fold_burdens[1][1])
    target_breach_penalty = 0.25
    burden_loss = abs(wf2022['mean_penalty_on_breach'] - target_breach_penalty)
    over_penalty_loss = max(0.0, wf2022['mean_penalty'] - 0.18) * 2.0
    oos_penalty_loss = wf2024['mean_penalty'] * 10.0
    complexity_loss = 0.01 * (position_penalty + drawdown_penalty)
    total_loss = burden_loss + over_penalty_loss + oos_penalty_loss + complexity_loss
    grid_rows.append({
        'mode': selected_mode,
        'min_feature_hits': selected_min_hits,
        'position_threshold': position_threshold,
        'position_penalty': position_penalty,
        'drawdown_threshold': drawdown_threshold,
        'drawdown_penalty': drawdown_penalty,
        'wf_2022_mean_penalty': wf2022['mean_penalty'],
        'wf_2022_mean_penalty_on_sequence': wf2022['mean_penalty_on_sequence'],
        'wf_2022_mean_penalty_on_breach': wf2022['mean_penalty_on_breach'],
        'wf_2022_active_penalty_rows': wf2022['active_penalty_rows'],
        'wf_2023_2024_mean_penalty': wf2024['mean_penalty'],
        'wf_2023_2024_active_penalty_rows': wf2024['active_penalty_rows'],
        'burden_loss': burden_loss,
        'over_penalty_loss': over_penalty_loss,
        'oos_penalty_loss': oos_penalty_loss,
        'complexity_loss': complexity_loss,
        'reward_recipe_loss': total_loss,
    })

reward_penalty_grid = pd.DataFrame(grid_rows).sort_values(['reward_recipe_loss', 'position_threshold']).reset_index(drop=True)
selected_recipe = reward_penalty_grid.iloc[0].to_dict()
reward_penalty_grid.to_csv(CARD / 'reward_penalty_grid.csv', index=False)
reward_penalty_grid.head(12)


,mode,min_feature_hits,position_threshold,position_penalty,drawdown_threshold,drawdown_penalty,wf_2022_mean_penalty,wf_2022_mean_penalty_on_sequence,wf_2022_mean_penalty_on_breach,wf_2022_active_penalty_rows,wf_2023_2024_mean_penalty,wf_2023_2024_active_penalty_rows,burden_loss,over_penalty_loss,oos_penalty_loss,complexity_loss,reward_recipe_loss
0,two_of_three,2,0.10,3.0,0.15,3.0,0.194891,0.259164,0.240877,157,0.0,0,0.009123,0.029783,0.0,0.060,0.098906
1,two_of_three,2,0.10,3.0,0.15,2.0,0.193122,0.256811,0.226133,157,0.0,0,0.023867,0.026244,0.0,0.050,0.100111
2,two_of_three,2,0.10,3.0,0.15,1.0,0.191353,0.254459,0.211390,157,0.0,0,0.038610,0.022706,0.0,0.040,0.101316
3,two_of_three,2,0.10,2.0,0.15,3.0,0.131697,0.175129,0.175328,157,0.0,0,0.074672,0.000000,0.0,0.050,0.124672
4,two_of_three,2,0.12,3.0,0.15,3.0,0.157543,0.209499,0.180877,155,0.0,0,0.069123,0.000000,0.0,0.060,0.129123
5,two_of_three,2,0.10,2.0,0.15,2.0,0.129928,0.172776,0.160585,157,0.0,0,0.089415,0.000000,0.0,0.040,0.129415
6,two_of_three,2,0.12,3.0,0.15,2.0,0.155774,0.207146,0.166133,155,0.0,0,0.083867,0.000000,0.0,0.050,0.133867
7,two_of_three,2,0.10,2.0,0.15,1.0,0.128158,0.170423,0.145841,157,0.0,0,0.104159,0.000000,0.0,0.030,0.134159
8,two_of_three,2,0.12,3.0,0.15,1.0,0.154004,0.204793,0.151390,155,0.0,0,0.098610,0.000000,0.0,0.040,0.138610
9,two_of_three,2,0.12,4.0,0.15,3.0,0.208288,0.276979,0.226426,155,0.0,0,0.023574,0.056576,0.0,0.070,0.150150


In [7]:
threshold_row = sequence_cap_audit[
    sequence_cap_audit['candidate'].eq('trend_break_any_cap10')
    & sequence_cap_audit['fold'].eq('wf_2022_train')
].iloc[0]
thresholds = json.loads(str(threshold_row['thresholds_json']))

reward_config_patch = {
    'sequence_risk_reward_enabled': True,
    'sequence_risk_position_penalty': float(selected_recipe['position_penalty']),
    'sequence_risk_position_threshold': float(selected_recipe['position_threshold']),
    'sequence_risk_drawdown_penalty': float(selected_recipe['drawdown_penalty']),
    'sequence_risk_drawdown_threshold': float(selected_recipe['drawdown_threshold']),
    'sequence_risk_min_feature_hits': int(selected_recipe['min_feature_hits']),
    'sequence_risk_anchor_slope_max': float(thresholds['anchor_slope_180']['threshold']),
    'sequence_risk_anchor_gap_max': float(thresholds['anchor_gap_180']['threshold']),
    'sequence_risk_trend_gap_min': float(thresholds['trend_gap_63']['threshold']),
}

selected_max_dd = float(selector_selection.get('selected_max_max_dd_pct', float('inf')))
selected_min_relative = float(selector_selection.get('selected_min_relative_return_pp', float('-inf')))
max_dd_gap = max(0.0, selected_max_dd - TARGET_MAX_DD_PCT)
full_gate_candidate_count = int(selector_selection.get('full_gate_candidate_count', 0) or 0)

if full_gate_candidate_count > 0:
    decision = 'reward_feature_screen_unneeded_candidate_review_no_promotion'
    next_mechanism = 'review_existing_full_gate_candidate_before_reward_training'
elif selected_mode == 'all' or float(selected_mode_row['wf_2022_breach_capture']) < 0.80:
    decision = 'sequence_risk_reward_feature_not_ready_no_promotion'
    next_mechanism = 'collect_narrower_sequence_risk_state_or_more_daily_paths'
elif max_dd_gap > 0.0 and float(selected_recipe['wf_2023_2024_mean_penalty']) == 0.0:
    decision = 'sequence_risk_reward_feature_ready_for_training_screen_no_promotion'
    next_mechanism = 'run_short_foldlocal_bc_or_ppo_screen_with_sequence_risk_reward_enabled_before_longer_ppo'
else:
    decision = 'sequence_risk_reward_feature_diagnostic_only_no_promotion'
    next_mechanism = 'recheck_reward_penalty_breadth_or target state before training'

selection = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'experiment': EXPERIMENT,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': [
        'cap10_sequence_risk_checkpoint_penalty_screen_v1',
        'cap10_sequence_risk_target_cap_bc_screen_v1',
        'cap10_bc_sequence_risk_state_diagnostic_v1',
    ],
    'role': 'diagnose',
    'changed_layer': ['reward', 'src_capability'],
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'decision': decision,
    'promotion_allowed': False,
    'training_allowed_from_this_card': False,
    'actor_retention': 'no actor bundle is promoted or retained by this notebook',
    'selected_source_candidate': selector_selection.get('selected_candidate_id'),
    'selected_source_min_relative_return_pp': selected_min_relative,
    'selected_source_max_dd_pct': selected_max_dd,
    'selected_source_max_dd_gap_pp': max_dd_gap,
    'selected_mode': selected_mode,
    'selected_min_feature_hits': selected_min_hits,
    'selected_mode_loss': float(selected_mode_row['mode_loss']),
    'selected_wf_2022_breach_capture': float(selected_mode_row['wf_2022_breach_capture']),
    'selected_wf_2022_sequence_share': float(selected_mode_row['wf_2022_sequence_share']),
    'selected_wf_2023_2024_sequence_share': float(selected_mode_row['wf_2023_2024_sequence_share']),
    'selected_reward_recipe_loss': float(selected_recipe['reward_recipe_loss']),
    'reward_config_patch': reward_config_patch,
    'anti_leakage_note': 'Feature family comes from prior diagnostics; fold-local thresholds are read from the target-cap audit and only used here to screen a default-off reward feature. This card does not reuse OOS thresholds as labels and does not train.',
    'promotion_blocker': 'Reward feature is only a training objective recipe; promotion still requires fixed-split and walk-forward validation after a trained candidate exists.',
    'next_mechanism': next_mechanism,
    'source_decisions': source_context,
    'external_sources': selector_selection.get('external_sources', []),
    'sidecars': [
        'sequence_risk_mode_scores.csv',
        'sequence_risk_mode_summary.csv',
        'reward_penalty_grid.csv',
        'selection.json',
        'summary.json',
    ],
}

summary = {
    'decision': decision,
    'selection': selection,
    'selected_mode': json_safe(selected_mode_row),
    'selected_recipe': json_safe(selected_recipe),
    'reward_config_patch': reward_config_patch,
    'mode_summary': json_safe(mode_summary.to_dict(orient='records')),
    'reward_penalty_grid_top': json_safe(reward_penalty_grid.head(12).to_dict(orient='records')),
    'source_context': source_context,
    'conclusion': [
        f"Selected sequence mode `{selected_mode}` with min_feature_hits={selected_min_hits}; wf_2022 breach_capture={selected_mode_row['wf_2022_breach_capture']:.2f}, sequence_share={selected_mode_row['wf_2022_sequence_share']:.3f}, wf_2023_2024 sequence_share={selected_mode_row['wf_2023_2024_sequence_share']:.3f}.",
        f"Selected reward recipe position_threshold={selected_recipe['position_threshold']:.2f}, position_penalty={selected_recipe['position_penalty']:.2f}, drawdown_penalty={selected_recipe['drawdown_penalty']:.2f}.",
        f"Source candidate still misses MaxDD by {max_dd_gap:.2f}pp, so no promotion or actor retention is allowed.",
        f"Next action: {next_mechanism}.",
    ],
}

(CARD / 'selection.json').write_text(json.dumps(json_safe(selection), indent=2, sort_keys=True) + '\n')
(CARD / 'summary.json').write_text(json.dumps(json_safe(summary), indent=2, sort_keys=True) + '\n')
selection


{'created_at': '2026-05-26T00:59:07.598458+00:00',
 'experiment': 'sequence_risk_reward_feature_screen_v1',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': ['cap10_sequence_risk_checkpoint_penalty_screen_v1',
  'cap10_sequence_risk_target_cap_bc_screen_v1',
  'cap10_bc_sequence_risk_state_diagnostic_v1'],
 'role': 'diagnose',
 'changed_layer': ['reward', 'src_capability'],
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 15.0,
 'decision': 'sequence_risk_reward_feature_ready_for_training_screen_no_promotion',
 'promotion_allowed': False,
 'training_allowed_from_this_card': False,
 'actor_retention': 'no actor bundle is promoted or retained by this notebook',
 'selected_source_candidate': 'sequence_target_cap_bc::baseline_no_sequence_cap',
 'selected_source_min_relative_return_pp': 30.5011159708892,
 'selected_source_max_dd_pct': 18.39642924161966,
 'selected_source_max_dd_gap_pp': 3.3964292416196606,
 'selected_mode': 'two_of_thre

In [8]:
conclusion_md = f"""
## Runtime conclusion

Decision: `{decision}`.

Selected mode: `{selected_mode}` / `sequence_risk_min_feature_hits={selected_min_hits}`.

Selected reward config patch:
```json
{json.dumps(reward_config_patch, indent=2, sort_keys=True)}
```

Key evidence:
- Source candidate: `{selector_selection.get('selected_candidate_id')}`.
- Source candidate max MaxDD: `{selected_max_dd:.2f}%`, gap `{max_dd_gap:.2f}pp` vs `15%` target.
- `wf_2022` sequence breach capture: `{selected_mode_row['wf_2022_breach_capture']:.2f}`.
- `wf_2022` sequence share: `{selected_mode_row['wf_2022_sequence_share']:.3f}`.
- `wf_2023_2024` sequence share: `{selected_mode_row['wf_2023_2024_sequence_share']:.3f}`.

Interpretation:
- The prior post-hoc target caps did not pass robustly, so this card keeps the mechanism as a default-off reward feature.
- `two_of_three` keeps full `wf_2022` breach capture while avoiding the broader `any` trigger; `all` is too narrow and misses the drawdown breach.
- The selected recipe is ready for a short fold-local BC/PPO training screen, not for promotion.

Next action: `{next_mechanism}`.

Artifacts: `sequence_risk_mode_scores.csv`, `sequence_risk_mode_summary.csv`, `reward_penalty_grid.csv`, `selection.json`, `summary.json`.
"""

display(Markdown(conclusion_md))
print('\n'.join(summary['conclusion']))



## Runtime conclusion

Decision: `sequence_risk_reward_feature_ready_for_training_screen_no_promotion`.

Selected mode: `two_of_three` / `sequence_risk_min_feature_hits=2`.

Selected reward config patch:
```json
{
  "sequence_risk_anchor_gap_max": -0.12578648457075514,
  "sequence_risk_anchor_slope_max": -0.010676839689780171,
  "sequence_risk_drawdown_penalty": 3.0,
  "sequence_risk_drawdown_threshold": 0.15,
  "sequence_risk_min_feature_hits": 2,
  "sequence_risk_position_penalty": 3.0,
  "sequence_risk_position_threshold": 0.1,
  "sequence_risk_reward_enabled": true,
  "sequence_risk_trend_gap_min": 0.1856199329014594
}
```

Key evidence:
- Source candidate: `sequence_target_cap_bc::baseline_no_sequence_cap`.
- Source candidate max MaxDD: `18.40%`, gap `3.40pp` vs `15%` target.
- `wf_2022` sequence breach capture: `1.00`.
- `wf_2022` sequence share: `0.752`.
- `wf_2023_2024` sequence share: `0.000`.

Interpretation:
- The prior post-hoc target caps did not pass robustly, so this card keeps the mechanism as a default-off reward feature.
- `two_of_three` keeps full `wf_2022` breach capture while avoiding the broader `any` trigger; `all` is too narrow and misses the drawdown breach.
- The selected recipe is ready for a short fold-local BC/PPO training screen, not for promotion.

Next action: `run_short_foldlocal_bc_or_ppo_screen_with_sequence_risk_reward_enabled_before_longer_ppo`.

Artifacts: `sequence_risk_mode_scores.csv`, `sequence_risk_mode_summary.csv`, `reward_penalty_grid.csv`, `selection.json`, `summary.json`.


Selected sequence mode `two_of_three` with min_feature_hits=2; wf_2022 breach_capture=1.00, sequence_share=0.752, wf_2023_2024 sequence_share=0.000.
Selected reward recipe position_threshold=0.10, position_penalty=3.00, drawdown_penalty=3.00.
Source candidate still misses MaxDD by 3.40pp, so no promotion or actor retention is allowed.
Next action: run_short_foldlocal_bc_or_ppo_screen_with_sequence_risk_reward_enabled_before_longer_ppo.
